# TRF3 — Tribunal Regional Federal da 3ª Região

Public process consultation (`cpopg`) for the federal courts under the third
region (SP and MS), via the PJe `ConsultaPublica` system at
[pje1g.trf3.jus.br/pje/](https://pje1g.trf3.jus.br/pje/ConsultaPublica/listView.seam).

| Feature | Available |
|---------|-----------|
| cpopg   | Yes       |
| cposg   | No        |
| cjsg    | No        |
| cjpg    | No        |

## Looking up a single process

In [1]:
import juscraper as jus

trf3 = jus.scraper("trf3")

In [2]:
df = trf3.cpopg("5005946-09.2025.4.03.6324")
print(df.shape)
df[["id_cnj", "processo", "classe", "data_distribuicao"]]

TRF3 cpopg: 100%|██████████| 1/1 [00:01<00:00,  1.08s/it]

(1, 12)


,id_cnj,processo,classe,data_distribuicao
0,50059460920254036324,5005946-09.2025.4.03.6324,PROCEDIMENTO DO JUIZADO ESPECIAL CÍVEL (436),29/09/2025


## Available columns

In [3]:
df.columns.tolist()

['processo',
 'data_distribuicao',
 'classe',
 'assunto',
 'jurisdicao',
 'orgao_julgador',
 'endereco_orgao',
 'polo_ativo',
 'polo_passivo',
 'movimentacoes',
 'documentos',
 'id_cnj']

The first three columns are the canonical scalars; the trailing four are
list-typed and carry the nested arrays (parties, movements, attached
documents).

## Inspecting movements

In [4]:
movs = df.iloc[0]["movimentacoes"]
print(f"{len(movs)} events recorded")
for m in movs[:3]:
    print(f"  {m['data']} - {m['descricao']}")

11 events recorded
  29/04/2026 12:51:15 - Remetidos os Autos ao CEJUSC ou Centros de Conciliação/Mediação da Subseção Judiciária
  29/04/2026 12:37:09 - Expedida/certificada a intimação eletrônica
  29/04/2026 12:37:08 - Proferido despacho de mero expediente


## Inspecting parties

In [5]:
print("Polo ativo:")
for p in df.iloc[0]["polo_ativo"]:
    print(f"  - {p['participante']}")

print()
print("Polo passivo:")
for p in df.iloc[0]["polo_passivo"]:
    print(f"  - {p['participante']}")

Polo ativo:
  - JOAO PAULO NASCIMENTO DA SILVA - CPF: 003.XXX.XXX-XX (AUTOR)

Polo passivo:
  - CAIXA ECONOMICA FEDERAL - CEF - CNPJ: 00.3XX.XXX/XXXX-XX (REU) Departamento Jurídico - Caixa Econômica Federal


## Looking up multiple processes at once

In [6]:
cnjs = [
    "50059460920254036324",
    "50061271020254036324",
    "50035362120254036342",
]
df_batch = trf3.cpopg(cnjs)
df_batch[["id_cnj", "processo", "classe"]]


TRF3 cpopg:   0%|          | 0/3 [00:00<?, ?it/s]


TRF3 cpopg:  33%|███▎      | 1/3 [00:01<00:02,  1.42s/it]


TRF3 cpopg:  67%|██████▋   | 2/3 [00:03<00:01,  1.63s/it]


TRF3 cpopg: 100%|██████████| 3/3 [00:04<00:00,  1.30s/it]


TRF3 cpopg: 100%|██████████| 3/3 [00:04<00:00,  1.37s/it]

,id_cnj,processo,classe
0,50059460920254036324,5005946-09.2025.4.03.6324,PROCEDIMENTO DO JUIZADO ESPECIAL CÍVEL (436)
1,50061271020254036324,5006127-10.2025.4.03.6324,PROCEDIMENTO DO JUIZADO ESPECIAL CÍVEL (436)
2,50035362120254036342,5003536-21.2025.4.03.6342,PROCEDIMENTO DO JUIZADO ESPECIAL CÍVEL (436)


## Handling processes the public portal cannot return

When a CNJ does not surface in the public consultation (sigilo, archived,
or simply not found), the row carries only `id_cnj`. The other columns
come back as `None`/`NaN`, so callers can still distinguish "looked up
but missing" from "never tried".

In [7]:
import pandas as pd

df_missing = trf3.cpopg("00000000020994030000")
print("processo:", df_missing.iloc[0].get("processo"))
print("classe:", df_missing.iloc[0].get("classe"))


TRF3 cpopg:   0%|          | 0/1 [00:00<?, ?it/s]


TRF3 cpopg: 100%|██████████| 1/1 [00:00<00:00,  8.34it/s]


TRF3 cpopg: 100%|██████████| 1/1 [00:00<00:00,  8.19it/s]

processo: None
classe: None


## Splitting download from parse

`cpopg` is a thin wrapper over `cpopg_download` (raw HTML) +
`cpopg_parse` (HTML → DataFrame). Splitting them is useful when you
want to cache the raw HTMLs to disk before processing.

In [8]:
htmls = trf3.cpopg_download("5005946-09.2025.4.03.6324")
print(f"got {len(htmls)} HTML(s), {len(htmls[0])} chars")

df_again = trf3.cpopg_parse(htmls, ["50059460920254036324"])
df_again[["id_cnj", "processo", "data_distribuicao"]]


TRF3 cpopg:   0%|          | 0/1 [00:00<?, ?it/s]


TRF3 cpopg: 100%|██████████| 1/1 [00:00<00:00,  2.43it/s]


TRF3 cpopg: 100%|██████████| 1/1 [00:00<00:00,  2.43it/s]

got 1 HTML(s), 85297 chars


,id_cnj,processo,data_distribuicao
0,50059460920254036324,5005946-09.2025.4.03.6324,29/09/2025
